# Multi-grid Methods
The multi-grid methods can be used to accelerate the convergence of a traditional relaxation method, such as Jacobi or Gauss-Seidel, to solve elliptic equations of the form $\mathcal{L}u = f$ (where $\mathcal{L}$ is the generalized differential operator). The essence of this method lies in the fact that relaxation methods converge faster on coarser grids, which can then be used to find an exact solution and that is interpolated back to the fine grid and applied as an initial condition. Multi-grid methods can be used in conjunction with several discretization techniques, such as finite element, finite volume, Jacobi, Gauss-Seidel and Successive Over Relaxation (SOR) methods. In contrast to other methods, multi-grid methods can be used with arbitrary grids and boundary conditions, and they do not require the separability of equations or other special properties of the equation.

## Algorithm
The multi-grid method follows the steps below:
1. Discretize the elliptic equation with a fine grid size $h$. For a initial guess solution $\tilde{u}_h$, define the residual vector $d_h$:
$$d_h = \mathcal{L}_h\tilde{u}_h - f_h$$
2. If the desired solution is $u_h$, we can define the correction to the guess $\Delta u_h = u_h-\tilde{u}_h$. This gives us the matrix equation:
$$\mathcal{L}_h\Delta u_h = - d_h$$
3. Run a few iterations of Gauss-Seidel (or other relaxation methods) to "smoothen" the correction error. 
4. Coarse-grain the equation using the restriction operator $\mathcal{R}$ to transform from the fine grid to the coarse grid $\mathcal{L}_h\Delta u_h = - d_h \rightarrow \mathcal{L}_H\Delta u_H = - d_H$. For a 2D problem with $H=2h$, the restriction matrix is: 
$$\mathcal{R} = \begin{bmatrix} 1/16 & 1/8 & 1/16 \\ 1/8 & 1/4 & 1/8 \\ 1/16 & 1/8 & 1/16 \end{bmatrix}$$
5. Solve for $v_H$ using matrix inversion at the coarse grid and interpolate using the prolongation operator $\mathcal{P}$ to transform back to the fine grid $v_H \rightarrow v_h$. The prolongation matrix for a 2D problem with $h=H/2$ is:
$$\mathcal{P} = \begin{bmatrix} 1/4 & 1/2 & 1/4 \\ 1/2 & 1 & 1/2 \\ 1/4 & 1/2 & 1/4 \end{bmatrix}$$
6. Use $u_h^{new} = \tilde{u}_h + v_h$ as the new initial solution and iteratively repeat the above steps until $u_h$ converges within a desired tolerance. Each round of this process is known as a V-cycle. 

In certain cases, it may be beneficial to interpolate only part of the way up before smoothening via Gauss-Seidel and coarse-graining to find another solution, before going back to the finest level. This sequence is called a W-cycle instead of a V-cycle. This entire process relies on a few key aspects, discussed below. 

## Error Components
At each iteration of a relaxation method (such as Gauss-Seidel), if we were to track the spatial variation of the numerical error of the solution, we would find that there is a large local variation in error, composed of what we call high-frequency components of the error. However, within a few iterations we would notice that these local differences are quickly smoothened out, leaving a much more even error distribution, composed of the low-frequency components. Usually, at any given iteration the error distribution is a combination of both of these components, but while relaxation methods are good are damping out high-frequency errors, they take very long to reduce low-frequency errors.

This is where the multi-grid method becomes useful, since after coarse-graining, a low-frequency error distribution is transformed into a high-frequency one, which can subsequently be "smoothened" by another round of Gauss-Seidel. If we fail to smoothen the error distribution before coarse-graining, the coarse-grained picture may not capture the shape of the fine-grained variation, leading to aliasing error. Hence, Gauss-Seidel is used as a smoother rather than a solver in the multi-grid method. 

## Residuals
Since we do not usually have prior knowledge of the analytical solution of a problem, we cannot determine the absolute numerical error at each iteration. Instead, we use the correction $\Delta u$ to assess the convergence of our solution since it behaves in a similar manner. Hence, it can be smoothened using Gauss-Seidel and transferred between grids without aliasing errors. 



# Spectral Methods
Spectral methods determine the solution for a PDE by reframing the problem in spectral space, thereby making the aim to determine the coefficients of the series of orthogonal basis functions. For a second-order equations in 3D, increasing the spatial resolution by a factor of 2 in each dimension requires 8 times as many grid points, and typically improves the error by a factor of 4. On the other hand, a similar increase in resolution in a spectral method can improve the error by a factor of $10^6$.

Consider the 2D heat equation over the interval $(x,y) \in [0, 2\pi]\times[0, 2\pi]$. 

$$\frac{\partial u(x,y,t)}{\partial t}=\nabla^2 u(x,y,t) = \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2}$$

With periodic boundary conditions $u(x,y,t) = u(x+2\pi,y,t)$ and $u(x,y,t) = (x,y+2\pi,t)$ and initial condition $u(x,y,t=0)=f(x,y)$. Since the boundary conditions are periodic, it is natural to choose fourier modes as our orthogonal basis functions. In 2D, the fourier series of $u$ is:  

$$u(x,y,t) = \sum_{k=-\infty}^\infty \sum_{l=-\infty}^\infty \hat{u}_{k,l}(t) e^{i(kx+ly)}$$

Computing the spatial derivatives:  
$$\frac{\partial^2 u}{\partial x^2} = -k^2\hat{u}_{k,l}(t) e^{i(kx+ly)}$$
$$\frac{\partial^2 u}{\partial y^2} = -l^2\hat{u}_{k,l}(t) e^{i(kx+ly)}$$

Putting this back into the differential equation: 
$$\frac{d\hat{u}_{k,l}}{dt} = -(k^2+l^2)\hat{u}_{k,l}$$
$$\implies \hat{u}_{k,l}(t) = \hat{u}_{k,l}(0)e^{-(k^2+l^2)t}$$

Comparing with the initial condition: 

$$u(x,y,0) = f(x,y) \implies \sum_{k=-\infty}^\infty \sum_{l=-\infty}^\infty \hat{u}_{k,l}(0) e^{i(kx+ly)} = \sum_{k=-\infty}^\infty \sum_{l=-\infty}^\infty \hat{f}_{k,l} e^{i(kx+ly)}$$
$$\implies \hat{u}_{k,l}(0) = \hat{f}_{k,l}$$

Hence, the coefficients are given by $\hat{u}_{k,l}(t) = \hat{f}_{k,l}e^{-(k^2+l^2)t}$. We can determine $\hat{f}_{k, l}$ from the inverse fourier transform:

$$\hat{f}_{k, l} = \frac{1}{(2\pi)^2} \int_{0}^{2\pi} \int_{0}^{2\pi} f(x,y) e^{-i(kx+ly)} dxdy$$

This is found by discretizing space as $x_j=j\Delta$ and $y_m = m\Delta$ where $j,m=0,1,\ldots N$, and therefore computing: 

$$\hat{f}_{k,l} = \frac{1}{N^2} \sum_{j=0}^{N} \sum_{m=0}^{N} f_{j,m} e^{-\frac{2\pi i}{N} (kj+lm)}$$

We can put these back into the fourier series to find the solution: 

$$u(x,y,t) = \sum_{k=-\infty}^\infty \sum_{l=-\infty}^\infty \hat{f}_{k,l}e^{-(k^2+l^2)t} e^{i(kx+ly)}$$